In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'e:\\Alekhya AEE\\Chatbot'

In [3]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
def load_pdf_file(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [5]:
extracted_data=load_pdf_file(data='data/')

Ignoring wrong pointing object 2 65536 (offset 0)
Ignoring wrong pointing object 83 65536 (offset 0)
Ignoring wrong pointing object 89 65536 (offset 0)
Ignoring wrong pointing object 104 65536 (offset 0)
Ignoring wrong pointing object 120 65536 (offset 0)
Ignoring wrong pointing object 133 65536 (offset 0)
Ignoring wrong pointing object 146 65536 (offset 0)
Ignoring wrong pointing object 159 65536 (offset 0)
Ignoring wrong pointing object 178 65536 (offset 0)
Ignoring wrong pointing object 193 65536 (offset 0)
Ignoring wrong pointing object 208 65536 (offset 0)
Ignoring wrong pointing object 224 65536 (offset 0)
Ignoring wrong pointing object 240 65536 (offset 0)
Ignoring wrong pointing object 256 65536 (offset 0)
Ignoring wrong pointing object 272 65536 (offset 0)
Ignoring wrong pointing object 288 65536 (offset 0)
Ignoring wrong pointing object 304 65536 (offset 0)
Ignoring wrong pointing object 318 65536 (offset 0)
Ignoring wrong pointing object 325 65536 (offset 0)
Ignoring wrong p

In [6]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=5)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [7]:
text_chunks=text_split(extracted_data)
len(text_chunks)

1666

In [8]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [9]:
embeddings=download_hugging_face_embeddings()

C:\Users\admin\AppData\Local\Temp\ipykernel_8396\1780091351.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
c:\Users\admin\miniconda3\envs\rcc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
query_result=embeddings.embed_query("Hello world")
print (len(query_result))

384


In [11]:
from pinecone import Pinecone, ServerlessSpec
pc=Pinecone(api_key= 'pcsk_4pdyoQ_eFVkeS49u1F2gSTMKBhuJbpQwf4C5NTkGf7QXmomQTNoLKfJNPnsDfJxaTfFGy' )
index_name = "rccbot"

pc.create_index(
    name=index_name,
    dimension=384, # Replace with your model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-01', 'X-Cloud-Trace-Context': 'f92dd98bd7bc77eb29742dd526dc0209', 'Date': 'Thu, 13 Mar 2025 05:43:48 GMT', 'Server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [12]:
from dotenv import load_dotenv
import os

# Load the environment variables from the .env file
load_dotenv()

import os
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [ ]:
GROQ_API_KEY

'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'

In [15]:
import os
os.environ['PINECONE_API_KEY']=PINECONE_API_KEY
os.environ['GROQ_API_KEY']=GROQ_API_KEY

In [16]:
from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [19]:
docsearch

In [18]:
retriever=docsearch.as_retriever(search_type='similarity',search_kwargs={"k":3})

In [19]:
retrieved_docs=retriever.invoke("What is the curing period?")

In [20]:
retrieved_docs

[Document(id='9379d4bc-3390-48a5-a844-9834ec2ee63a', metadata={'author': 'Bureau of Indian Standards', 'creationdate': '2013-09-05T09:03:05-07:00', 'creator': 'pdftk 1.44 - www.pdftk.com', 'moddate': '2013-09-05T09:03:05-07:00', 'page': 39.0, 'page_label': '40', 'producer': 'itext-paulo-155 (itextpdf.sf.net-lowagie.com)', 'source': 'data\\is.456.2000.pdf', 'subject': 'Published Under the Right to Information Act', 'title': 'IS 456 (2000): Plain and Reinforced Concrete - Code of Practice', 'total_pages': 114.0}, page_content='Approved curing compoundsmay be used in lieu of\nmoist curing withthepermissionof theengineer..in\xad\ncharge.Suchcompoundsshallbeappliedtoallexposed\nsurfacesof the concrete as soon aspossibleafter the'),
 Document(id='558a0d33-2195-4a04-9e60-98ed9d0243a3', metadata={'author': 'Bureau of Indian Standards', 'creationdate': '2013-09-05T09:03:05-07:00', 'creator': 'pdftk 1.44 - www.pdftk.com', 'moddate': '2013-09-05T09:03:05-07:00', 'page': 35.0, 'page_label': '36', 

In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

In [22]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
system_prompt=(
"You are an assistant for question-answering tasks.  "
"Use the following pieces of retrieved context to answer the question"
"Use three sentences to keep the answer concise"
"\n\n"
"{context}"
)

In [23]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [24]:
question_answer_chain=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)

In [25]:
response=rag_chain.invoke({"input":"What is the minimum curing period of concrete?"})
response['answer']

'The minimum curing period for concrete is not explicitly stated for all types, but for ordinary Portland cement, it is at least 10 days. When mineral admixtures or blended cements are used, the minimum curing period may be extended to 14 days. In cases of extreme weather conditions, such as hot and dry environments, the curing period should not be less than 10 days.'

In [ ]:
import pdfplumber
import os
import pinecone
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Pinecone
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq

# 1. Extract tables using pdfplumber from a PDF
def extract_pdf_tables(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        all_tables = []
        for page in pdf.pages:
            tables = page.extract_tables()
            all_tables.extend(tables)
        return all_tables

# 2. Split extracted text into manageable chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=5)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

# 3. Download Groq Embeddings (Here you are using the Groq API for embedding)
def download_groq_embeddings():
    embeddings = Embeddings(model="llama-3.3-70b-versatile", temperature=0, groq_api_key=os.getenv("GROQ_API_KEY"))
    return embeddings

# 4. Initialize Pinecone and Create a Vector Store
def initialize_pinecone():
    # Initialize Pinecone
    pinecone.init(api_key=os.getenv("PINECONE_API_KEY"), environment="us-west1-gcp")  # Replace with your region
    # Create Pinecone index if it doesn't exist
    if "is456" not in pinecone.list_indexes():
        pinecone.create_index("is456", dimension=384)  # Use appropriate dimension size
    index = pinecone.Index("is456")
    return index

def create_vectorstore(text_chunks, embeddings, index):
    # Convert text chunks into embeddings and store them in Pinecone
    vectors = [embeddings.embed(text) for text in text_chunks]
    metadata = [{"text": text} for text in text_chunks]
    index.upsert(vectors=zip(range(len(vectors)), vectors, metadata))
    return Pinecone(index=index, embedding_function=embeddings.embed)

# 5. Create Retrieval QA Chain using Groq
def create_qa_chain(vectorstore):
    qa_chain = RetrievalQA.from_chain_type(
        llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0, groq_api_key=os.getenv("GROQ_API_KEY")),
        chain_type="map_reduce",  # You can experiment with other chain types like "stuff", "map_rerank"
        retriever=vectorstore.as_retriever()
    )
    return qa_chain

# 6. Process PDF, Extract Knowledge, and Answer Queries
def process_pdf_and_query(pdf_path):
    # Step 1: Extract tables from the PDF
    tables = extract_pdf_tables(pdf_path)
    
    # Convert extracted tables to text (you can format them better if needed)
    extracted_data = []
    for table in tables:
        for row in table:
            extracted_data.append(" | ".join(row))
    
    # Step 2: Split the extracted data into manageable chunks
    text_chunks = text_split(extracted_data)
    
    # Step 3: Download Groq Embeddings
    embeddings = download_groq_embeddings()
    
    # Step 4: Initialize Pinecone and create the vector store
    index = initialize_pinecone()
    
    # Step 5: Create the vector store in Pinecone
    vectorstore = create_vectorstore(text_chunks, embeddings, index)
    
    # Step 6: Set up the RetrievalQA chain
    qa_chain = create_qa_chain(vectorstore)
    
    # Example Query (you can replace this with any user query)
    query = "What is the minimum cement content for M25 concrete in moderate conditions?"
    
    # Step 7: Get the answer from the RetrievalQA chain
    answer = qa_chain.run(query)
    return answer

# Example: Running the process
pdf_path = "E:\Alekhya AEE\Chatbot\data\is.456.2000.pdf"  # Replace with the path to your IS 456 PDF
answer = process_pdf_and_query(pdf_path)
print(answer)


FileNotFoundError: [Errno 2] No such file or directory: 'path_to_your_IS_456_2000.pdf'